# BERT-base — 200k Kaggle run

This is the Kaggle execution notebook for one experiment. Run it with a GPU accelerator after attaching the prepared split dataset.

Settings:

- model: `bert-base-uncased`
- train / validation / test: 200,000 / 20,000 / 178,083
- epochs: 2
- max length: 128
- train / evaluation batch size: 16 / 32
- learning rate: 2e-5
- weight decay: 0.01
- warmup ratio: 0.0
- seed: 42
- approximate runtime: about 1 hour 40 minutes

## 1. Kaggle environment

Enable a GPU accelerator before running the notebook.

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import inspect
import json
import platform
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

MODEL_NAME = 'bert-base-uncased'
RUN_NAME = 'bert_base_200k'
EXPERIMENT = '200k'

SEED = 42
EPOCHS = 2
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.0
LOGGING_STEPS = 200

TEXT_COLUMN = "comment_text"
LABEL_COLUMN = "label"
ID_COLUMN = "id"

DATA_DIR = Path("/kaggle/input/datasets/shirley124123/test-20k")
TRAIN_PATH = DATA_DIR / "experiment_train_200k.csv"
VALIDATION_PATH = DATA_DIR / "experiment_validation_20k.csv"
TEST_PATH = DATA_DIR / "test.csv"
EXPECTED_ROWS = {"train": 200_000, "validation": 20_000, "test": 178_083}

OUTPUT_DIR = Path("/kaggle/working/results/fixed_200k/bert_base")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
RESULTS_ZIP = Path("/kaggle/working") / f"{RUN_NAME}_results.zip"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle before training.")

print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Data:", DATA_DIR)
print("Output:", OUTPUT_DIR)

## 2. Load the prepared splits

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

frames = {"train": train_df, "validation": validation_df, "test": test_df}
for split_name, frame in frames.items():
    required = {ID_COLUMN, TEXT_COLUMN, LABEL_COLUMN}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing)}")
    assert len(frame) == EXPECTED_ROWS[split_name]
    assert frame[ID_COLUMN].is_unique
    assert set(frame[LABEL_COLUMN].dropna().unique()).issubset({0, 1})

print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

## 3. Tokenise the text

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_frame(frame):
    prepared = frame.copy()
    prepared[TEXT_COLUMN] = prepared[TEXT_COLUMN].fillna("").astype(str)
    prepared[LABEL_COLUMN] = prepared[LABEL_COLUMN].astype("int8")
    return prepared

train_df = prepare_frame(train_df)
validation_df = prepare_frame(validation_df)
test_df = prepare_frame(test_df)
test_metadata = test_df.copy()

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

def tokenise(batch):
    return tokenizer(batch[TEXT_COLUMN], truncation=True, max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenise, batched=True, desc="Tokenising train")
validation_dataset = validation_dataset.map(tokenise, batched=True, desc="Tokenising validation")
test_dataset = test_dataset.map(tokenise, batched=True, desc="Tokenising test")

train_dataset = train_dataset.rename_column(LABEL_COLUMN, "labels")
validation_dataset = validation_dataset.rename_column(LABEL_COLUMN, "labels")
test_dataset = test_dataset.rename_column(LABEL_COLUMN, "labels")

model_columns = [
    column for column in ["input_ids", "attention_mask", "token_type_ids", "labels"]
    if column in train_dataset.column_names
]
train_dataset.set_format(type="torch", columns=model_columns)
validation_dataset.set_format(type="torch", columns=model_columns)
test_dataset.set_format(type="torch", columns=model_columns)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 4. Train the model

In [ ]:
def metric_values(labels, predictions):
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "toxic_precision": float(precision_score(labels, predictions, zero_division=0)),
        "toxic_recall": float(recall_score(labels, predictions, zero_division=0)),
        "toxic_f1": float(f1_score(labels, predictions, zero_division=0)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
    }

def compute_metrics(result):
    predictions = np.argmax(result.predictions, axis=1)
    return metric_values(result.label_ids, predictions)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "num_train_epochs": EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "logging_strategy": "steps",
    "logging_steps": LOGGING_STEPS,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "toxic_f1",
    "greater_is_better": True,
    "fp16": True,
    "seed": SEED,
    "data_seed": SEED,
    "report_to": [],
}
evaluation_key = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
    else "evaluation_strategy"
)
training_kwargs[evaluation_key] = "epoch"


training_args = TrainingArguments(**training_kwargs)
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": validation_dataset,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
start_time = time.time()
trainer.train()
training_seconds = time.time() - start_time

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"Training time: {training_seconds / 60:.2f} minutes")

## 5. Evaluate the fixed test set

In [ ]:
prediction_output = trainer.predict(test_dataset)
logits = prediction_output.predictions
if isinstance(logits, tuple):
    logits = logits[0]

predictions = np.argmax(logits, axis=1).astype("int8")
labels = prediction_output.label_ids.astype("int8")

metrics = metric_values(labels, predictions)
tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
metrics.update({
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "training_seconds": float(training_seconds),
    "test_loss": float(prediction_output.metrics.get("test_loss", np.nan)),
})
print(json.dumps(metrics, indent=2))

## 6. Save and download the results

In [ ]:
prediction_frame = test_metadata[["id"]].copy()
prediction_frame["predicted_label"] = predictions
prediction_frame.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
prediction_frame[(labels == 0) & (predictions == 1)].to_csv(
    OUTPUT_DIR / "false_positives.csv", index=False
)
prediction_frame[(labels == 1) & (predictions == 0)].to_csv(
    OUTPUT_DIR / "false_negatives.csv", index=False
)

(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(OUTPUT_DIR / "training_history.json").write_text(
    json.dumps(trainer.state.log_history, indent=2), encoding="utf-8"
)

configuration = {
    "model_name": MODEL_NAME,
    "run_name": RUN_NAME,
    "data_directory": str(DATA_DIR),
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
    "epochs": EPOCHS,
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "seed": SEED,
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(configuration, indent=2), encoding="utf-8")

report = classification_report(
    labels, predictions, target_names=["non_toxic", "toxic"], digits=4, zero_division=0
)
(OUTPUT_DIR / "classification_report.txt").write_text(report, encoding="utf-8")

matrix = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(matrix, display_labels=["non_toxic", "toxic"]).plot(
    ax=ax, values_format="d"
)
ax.set_title(f"{RUN_NAME} confusion matrix")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180)
plt.close(fig)

zip_path = shutil.make_archive(str(RESULTS_ZIP.with_suffix("")), "zip", root_dir=OUTPUT_DIR)
print("Results folder:", OUTPUT_DIR)
print("Download:", zip_path)